# Google cluster traces: certify greedy placement before invoking MILP

[Open in Colab](https://colab.research.google.com/github/crypticsymmetry/improved-giggle/blob/main/notebooks/google_cluster_gate.ipynb).

Run a CPU-only gate on about 4.5 MB of checksum-verified public records. Evaluate three integer greedy proposals, then obtain a conservative lower bound from the LP dual. When the integer lower bound matches the best verified greedy score, return that proposal with an optimality certificate. Otherwise invoke MILP and retain the best independently verified candidate.

The benchmark also solves a separate MILP reference for quality comparison. Avoided invocations count only the gated policy, not all solver calls in this evaluation. Timings are descriptive and do not establish a speedup. No private data, API key or GPU is needed.

In [ ]:
import importlib
import importlib.util
import inspect
import subprocess
import sys

needs_install = importlib.util.find_spec("assumption_ops") is None
if not needs_install:
    needs_install = importlib.util.find_spec("assumption_ops.cluster_holdout") is None
if not needs_install:
    from assumption_ops.cluster_holdout import run_cluster_holdout
    needs_install = "include_gated" not in inspect.signature(run_cluster_holdout).parameters
if needs_install:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--upgrade",
        "git+https://github.com/crypticsymmetry/improved-giggle.git",
    ])
    for name in list(sys.modules):
        if name == "assumption_ops" or name.startswith("assumption_ops."):
            del sys.modules[name]
    importlib.invalidate_caches()

import json
import os
from pathlib import Path
from pprint import pprint
from IPython.display import Markdown, display
from assumption_ops.cluster_data import download_cluster
from assumption_ops.cluster_holdout import (
    DEFAULT_WINDOWS, run_cluster_holdout, write_cluster_holdout_csv,
)
assert "include_gated" in inspect.signature(run_cluster_holdout).parameters

## Download verified inputs

Only the complete machine-events object and first task-events shard are fetched. Exact bytes and SHA-256 digests are verified before atomic writes. Set `ASSUMPTION_OPS_CLUSTER_DIR` for offline validation with the same pinned files.

In [ ]:
data_dir = Path(os.environ.get("ASSUMPTION_OPS_CLUSTER_DIR", "cluster_data"))
if os.environ.get("ASSUMPTION_OPS_CLUSTER_DIR"):
    paths = {
        "machines": data_dir / "google_machine_events.csv.gz",
        "tasks": data_dir / "google_task_events.csv.gz",
    }
else:
    paths = download_cluster(data_dir)
assert all(Path(path).is_file() for path in paths.values())
pprint({key: {"path": str(path), "bytes": Path(path).stat().st_size}
        for key, path in paths.items()})

## Fix the protocol before solving

Use the same one development, one validation and six holdout windows as the job-disjoint benchmark. Later windows exclude all jobs admitted earlier, including departed tasks. Use every 1, 2, 4 and 8 machine case, at most 64 admitted task identities, and a five-second MILP limit. Each snapshot has no previous placement and zero migration cost. No outcomes change these settings.

In [ ]:
WINDOWS = DEFAULT_WINDOWS
MACHINE_COUNTS = (1, 2, 4, 8)
MAX_TASKS = 64
TIME_LIMIT_SECONDS = 5
experiment_settings = {
    "machine_counts": MACHINE_COUNTS,
    "max_tasks": MAX_TASKS,
    "time_limit_seconds": TIME_LIMIT_SECONDS,
    "windows": [
        {"id": w.id, "split": w.split, "start_us": w.start_us, "end_us": w.end_us}
        for w in WINDOWS
    ],
}
pprint(experiment_settings)

In [ ]:
report = run_cluster_holdout(
    paths["machines"], paths["tasks"], windows=WINDOWS,
    machine_counts=MACHINE_COUNTS, max_tasks=MAX_TASKS,
    time_limit=TIME_LIMIT_SECONDS, include_gated=True,
)
pprint(report["protocol"])
print("Cohorts:", len(report["cohorts"]), "capacity cases:", len(report["cases"]))
assert report["controlled_model"] and report["real_source_records"]
assert not report["external_dispatch"]
pprint(report["environment"])

## Inspect quality and gated invocations by split

Empty cohorts remain visible and are excluded from case denominators. A gate certificate proves optimality for this integer model; a verified time-limited proposal without a certificate is only feasible. The separate MILP reference remains in the evaluation even when the gated policy avoids its own MILP call.

In [ ]:
seen_jobs = set()
cohort_rows = ["| Window | Split | Status | Active tasks | Admitted jobs |",
               "|---|---|---|---:|---:|"]
for cohort in report["cohorts"]:
    window = cohort["window"]
    jobs = set(cohort["metadata"]["selected_job_ids"])
    assert seen_jobs.isdisjoint(jobs), "A job appears in multiple cohorts"
    seen_jobs.update(jobs)
    cohort_rows.append(
        f"| {window['id']} | {window['split']} | {cohort['status']} | "
        f"{cohort['tasks']} | {len(jobs)} |"
    )
display(Markdown("\n".join(cohort_rows)))
rows = ["| Split | Cases | Gated MILP calls | Certificates | Routes | Mean regret to reference |",
        "|---|---:|---:|---:|---|---:|"]
for split, summary in report["summary"]["by_split"].items():
    values = summary["methods"]["lp_gated"]
    regret = values["mean_absolute_regret_to_milp"]
    shown_regret = "unavailable" if regret is None else f"{regret:.4g}"
    rows.append(
        f"| {split} | {values['comparisons']} | {values['milp_invocations']} | "
        f"{values['certified_optimal_cases']} | {values['routes']} | {shown_regret} |"
    )
display(Markdown("\n".join(rows)))

## Check every route, score and certificate

Greedy assignments are whole tasks, independently checked against CPU and memory limits. The gate uses an exact rational evaluation of a dual lower bound, with its integer ceiling, rather than a floating-point near-equality test. The fractional LP assignment is never rounded or dispatched. A missing certificate or a positive gap triggers MILP. If its verified incumbent is worse than greedy, the gate keeps greedy; a narrow no-incumbent time-limit fallback remains explicitly uncertified.

In [ ]:
rows = ["| Window | Machines | Route | MILP called | Certified | Score ↓ | Reference score ↓ | Integer lower bound |",
        "|---|---:|---|---|---|---:|---:|---:|"]
for case in report["cases"]:
    results = {r["method"]: r for r in case["comparison"]["results"]}
    result, reference = results["lp_gated"], results["milp"]
    metrics, costs = result["evaluation"]["metrics"], result["evaluation"]["costs"]
    gate = result["gate"]
    assert metrics["placed_tasks"] + metrics["pending_tasks"] == metrics["tasks"]
    assert metrics["used_cpu"] <= metrics["cpu_capacity"]
    assert metrics["used_memory"] <= metrics["memory_capacity"]
    assert costs["migration"] == 0
    if gate["route"] == "lp_certificate":
        assert not gate["milp_invoked"] and gate["certified_optimal"]
        assert gate["integer_lower_bound"] == costs["total"]
    rows.append(
        f"| {case['window_id']} | {case['machine_count']} | {gate['route']} | "
        f"{gate['milp_invoked']} | {gate['certified_optimal']} | {costs['total']} | "
        f"{reference['evaluation']['costs']['total']} | {gate['integer_lower_bound']} |"
    )
display(Markdown("\n".join(rows)))

## Export complete evidence

JSON preserves the fixed protocol, source hashes, cohorts, input fingerprints, proposals, evaluations and gate diagnostics. CSV retains all methods and gate route columns. Alternative scores are not an additive execution ledger.

In [ ]:
output_dir = Path("cluster_gate_results")
output_dir.mkdir(exist_ok=True)
json_path, csv_path = output_dir / "gate.json", output_dir / "gate.csv"
json_path.write_text(json.dumps(report, indent=2, allow_nan=False) + "\n", encoding="utf-8")
write_cluster_holdout_csv(report, csv_path)
print("Saved:", json_path.resolve(), csv_path.resolve())
pprint(report["limitations"])
if (importlib.util.find_spec("google") is not None
        and importlib.util.find_spec("google.colab") is not None):
    from google.colab import files
    files.download(str(json_path))
    files.download(str(csv_path))

## Interpretation limits

The windows share one trace and infrastructure; grid cases reuse their window's tasks. Job separation prevents identity reuse but does not establish statistical independence. This experiment performs no fitting, confidence intervals, hypothesis tests or adaptive selection.

Selected capacities do not represent known spare production capacity. Existing occupancy, unsupported affinity, disk, networking, durations, general machine attributes and actual ingestion delay are excluded, and overcommit is not modeled. Results measure feasible proposed scores, not observed production completion, latency or business ROI.

The development region was previously explored. This extension is reproducible, but it is not an untouched external test set or a preregistered study. With no previous plan, stability-aware best fit equals ordinary best fit here; their equality is not independent extra evidence.

The gate can certify a proposal without running its own MILP, but the complete benchmark runs a reference MILP and LP checks for other methods. Route counts therefore do not measure total solver calls avoided by this notebook. Wall times depend on ordering, warm caches and setup overhead; no production speedup is inferred.